In [1]:
!pip install /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
!pip install /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl

Processing /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
autograd is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Processing /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
  Preparing metadata (setup.py) ... done
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4031 sha256=94da7d973d468f94d704c780baef198b61cd40249dadc068ca1876a40b20429f
  Stored in directory: /root/.cache/pip/wheels/6b/b5/e0/4c79e15c0b5f2c15ecf613c720bb20daab20a666eb67135155
Successfully built autograd-gamma
Processing /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl


In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats
import tensorflow_decision_forests as tfdf

In [3]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [4]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [5]:
from lifelines import KaplanMeierFitter,NelsonAalenFitter
from lifelines.utils import concordance_index as ci_lifelines


def calculate_survival_probabilities(data, time_col, event_col):
    naf = NelsonAalenFitter()
    naf.fit(durations=data['efs_time'], event_observed=data['efs'])
    data['target'] = naf.cumulative_hazard_at_times(data['efs_time']).values
    data['target'] = data['target'] * -1
    return data

def preprocess_survival_data(df, time_col='efs_time', event_col='efs'):
    df = calculate_survival_probabilities(df, time_col, event_col)
    return df

#train = preprocess_survival_data(train)

In [6]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [7]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        #train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]] = train[i[0]].fillna(0)
        #train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        #test[i[0]]=encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))
    else:
        test[i[0]] = test[i[0]].fillna(0)
        #test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [8]:
train_x = train.drop(columns=['efs', 'efs_time'])
train_y = train['efs']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.1, random_state=100)

In [10]:
catboost = CatBoostRegressor(grow_policy='Depthwise',
                             iterations=1000,
                             eval_metric='RMSE',
                             random_state=0,
                             verbose=100)



cat_features = list(train_x.select_dtypes(include=['object', 'category']).columns)
    
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_test, y_test, cat_features=cat_features)
        
catboost.fit(train_pool, eval_set=(val_pool), verbose=50, early_stopping_rounds=100)

Learning rate set to 0.084886
0:	learn: 0.4933273	test: 0.4934958	best: 0.4934958 (0)	total: 131ms	remaining: 2m 11s
50:	learn: 0.4468201	test: 0.4522435	best: 0.4522435 (50)	total: 1.91s	remaining: 35.5s
100:	learn: 0.4394936	test: 0.4490463	best: 0.4490229 (98)	total: 3.58s	remaining: 31.9s
150:	learn: 0.4321990	test: 0.4473794	best: 0.4473794 (150)	total: 5.14s	remaining: 28.9s
200:	learn: 0.4267106	test: 0.4465555	best: 0.4465555 (200)	total: 6.58s	remaining: 26.1s
250:	learn: 0.4225606	test: 0.4459069	best: 0.4458636 (248)	total: 7.98s	remaining: 23.8s
300:	learn: 0.4193269	test: 0.4455974	best: 0.4455851 (294)	total: 9.36s	remaining: 21.7s
350:	learn: 0.4159389	test: 0.4454447	best: 0.4453427 (319)	total: 10.8s	remaining: 19.9s
400:	learn: 0.4135223	test: 0.4452953	best: 0.4451493 (382)	total: 12s	remaining: 18s
450:	learn: 0.4100498	test: 0.4454051	best: 0.4451493 (382)	total: 13.4s	remaining: 16.3s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.445149334
b

In [11]:
cat_pre_test=catboost.predict(test)
print(f'Concordance index (lifelines): {ci_lifelines(y_test, catboost.predict(X_test))}')

Concordance index (lifelines): 0.7577996807518387


In [12]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']
test_predictions = (cat_pre_test)
#((RandomF_pre_test/Fold)+(LGBM_pre_test/Fold)+(xgb_pre_test/Fold)+(cat_pre_test/Fold))/4
#model.predict(new_test_data)
#((RandomF_pre_test/10)+(LGBM_pre_test/5)+(xgb_pre_test/5)+(cat_pre_test/5))/4
test_predictions

array([0.19940398, 0.73259   , 0.00634044])

In [13]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.199404
1,28801,0.732590
2,28802,0.006340


In [14]:
submission.to_csv('submission.csv', index = False)